# Long-Context Evaluation

This notebook builds deterministic synthetic probes for long-context retrieval and source-selection. The goal is to separate a model's ability to preserve distant evidence from its ability to pattern-match short local cues.


## Motivation

Long-context modeling is not one problem. Some failures come from hard context-window limits, some from poor retrieval over distant tokens, and some from selecting the wrong evidence when nearby text conflicts with older text. Transformer-XL extends attention with recurrent segment memory, explicitly targeting fixed-context limitations [@transformer_xl_2019]. Memorizing Transformers add approximate external retrieval so rare but important evidence can be recovered without forcing every fact into the active attention window [@memorizing_transformers_2022]. Longformer is a useful contrast because its local-window plus sparse-global pattern shows how architectural bias can privilege nearby evidence unless special global routes are available [@longformer_2020].


## Hypothesis

If the task construction keeps the answer unique and removes natural-language shortcuts, then distance-conditioned exact-match accuracy should expose where a baseline forgets. A full-context oracle should stay flat across distance buckets. A fixed-window baseline should degrade once the answer-bearing span moves outside its visible region, and a near/far conflict task should show a distinct failure mode: preferring the visible distractor over the distant correct source.


## Baseline

We compare two deliberately simple baselines.

1. **Oracle parser:** returns the recorded synthetic answer and represents perfect evidence access.
2. **Fixed-window memory baseline:** succeeds on single-evidence tasks only when the answer-bearing span is within the last `W` context tokens. On conflicting-evidence tasks it returns the nearby distractor whenever the requested source is too far away.

These baselines are intentionally crude. The notebook is about evaluation design, not about claiming a realistic state-of-the-art model.


## Metric

The primary metric is exact-match accuracy. To make long-context behavior visible, we condition that accuracy on evidence distance:

- `distance_tokens`: number of context tokens between the answer-bearing span and the query.
- distance bucket accuracy: exact-match averaged only over examples whose support distance falls in the same bucket.

A flat aggregate score can hide the failure we care about. Bucketed metrics show whether accuracy collapses only when support moves from local to medium or long range.


## Contamination-resistant task construction

The tasks in this notebook are synthetic on purpose.

- Answers, keys, entities, and distractors are generated from reserved symbolic prefixes such as `PASS_`, `COPY_`, `KEY_`, `VAL_`, `FAR_`, and `NEAR_`.
- Every example is built so the correct answer string appears exactly once in the prompt.
- Distractors use different prefixes from the correct answer to avoid accidental substring collisions.
- There are no natural facts, names, or benchmark prompts to memorize, so success must come from the current context rather than pretraining leakage.
- Seeds are explicit and deterministic, which makes regression tests and notebook reruns stable.

That does not eliminate all shortcut risk, but it removes the most obvious contamination path: reusing a public benchmark or letting the answer leak multiple times in the same context.


## Mathematical derivation

For an example `i`, let `d_i` be the distance in tokens from the answer-bearing span to the query, and let `W` be the visible context window of a simple retrieval baseline.

For the single-evidence tasks in this notebook (passkey retrieval, delayed copy, repeated-key induction), the fixed-window baseline is correct exactly when the support lies inside the window:

$$
\hat{y}_i(W) =
\begin{cases}
 y_i, & d_i \le W \\
 \text{MISS}, & d_i > W
\end{cases}
$$

so the bucketed accuracy over a bucket `B` is

$$
A_B(W) = \frac{1}{|B|} \sum_{i \in B} \mathbf{1}[d_i \le W].
$$

For conflicting evidence, a nearby distractor and a far correct answer produce a different error surface. If the requested source is far and the window only retains the nearby conflicting record, then the prediction becomes the distractor rather than the true answer. That means distance-conditioned evaluation is not only about whether retrieval fails, but also about *how* it fails: omission versus confident source confusion.


In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.sequence_tasks import (
    distance_conditioned_accuracy,
    generate_conflicting_evidence_example,
    generate_delayed_copy_example,
    generate_passkey_retrieval_example,
    generate_repeated_key_induction_example,
)

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)


## PyTorch implementation

The helper module gives us four deterministic generators:

- passkey retrieval
- delayed copy
- repeated-key induction
- conflicting near/far evidence

Below we build a small evaluation suite, define the toy baselines, and compute distance-conditioned metrics with PyTorch tensors.


In [ ]:
BOUNDARIES = [16, 48, 96]


def make_suite() -> list:
    examples = []
    passkey_noise = [8, 24, 56, 104]
    delayed_noise = [8, 24, 56, 104]
    repeated_noise_pairs = [2, 5, 10, 18]
    conflict_noise = [8, 24, 56, 104]

    for offset, noise in enumerate(passkey_noise):
        for seed in range(4):
            examples.append(
                generate_passkey_retrieval_example(
                    seed=1000 + 100 * offset + seed,
                    noise_tokens=noise,
                )
            )
    for offset, delay in enumerate(delayed_noise):
        for seed in range(4):
            examples.append(
                generate_delayed_copy_example(
                    seed=2000 + 100 * offset + seed,
                    delay_tokens=delay,
                )
            )
    for offset, noise_pairs in enumerate(repeated_noise_pairs):
        for seed in range(4):
            examples.append(
                generate_repeated_key_induction_example(
                    seed=3000 + 100 * offset + seed,
                    noise_pairs=noise_pairs,
                    target_repetitions=3,
                )
            )
    for offset, noise in enumerate(conflict_noise):
        for seed in range(4):
            examples.append(
                generate_conflicting_evidence_example(
                    seed=4000 + 100 * offset + seed,
                    noise_tokens=noise,
                    correct_source='far',
                )
            )
    return examples


def oracle_predict(example) -> str:
    return example.answer



def fixed_window_predict(example, window_tokens: int) -> str:
    if example.task_name == 'conflicting_evidence':
        if example.distance_tokens <= window_tokens:
            return example.answer
        return str(example.metadata['distractor_answer'])
    if example.distance_tokens <= window_tokens:
        return example.answer
    return 'MISS'



def torch_bucket_accuracy(examples, predictions, boundaries):
    distances = torch.tensor([example.distance_tokens for example in examples], dtype=torch.int64)
    correct = torch.tensor(
        [prediction.strip() == example.answer.strip() for example, prediction in zip(examples, predictions)],
        dtype=torch.float32,
    )
    lowers = [0, *boundaries]
    uppers = [*boundaries, None]
    rows = []
    for lower, upper in zip(lowers, uppers):
        mask = distances >= lower
        if upper is not None:
            mask = mask & (distances < upper)
        count = int(mask.sum().item())
        accuracy = 0.0 if count == 0 else float(correct[mask].mean().item())
        label = f'[{lower}, inf)' if upper is None else f'[{lower}, {upper})'
        rows.append({'label': label, 'count': count, 'accuracy': accuracy})
    return rows



def format_rows(rows):
    header = "| Bucket | Count | Accuracy |\n|---|---:|---:|"
    body = "\n".join(
        f"| {row['label']} | {row['count']} | {row['accuracy']:.3f} |" for row in rows
    )
    return header + "\n" + body


suite = make_suite()
len(suite)


In [ ]:
sample_tasks = []
seen = set()
for example in suite:
    if example.task_name not in seen:
        seen.add(example.task_name)
        sample_tasks.append(example)

for example in sample_tasks:
    print(example.task_name)
    print('distance_tokens =', example.distance_tokens)
    print('prompt snippet =', example.prompt[:140] + '...')
    print('answer =', example.answer)
    print()


## Numerical checks

A research notebook should verify the data-generating process before interpreting any metric. We check three things:

1. the answer-bearing span really points to the unique answer,
2. a repeated seed reproduces the same example, and
3. the helper's bucket metric matches a direct PyTorch aggregation.


In [ ]:
for generator, kwargs in [
    (generate_passkey_retrieval_example, {'noise_tokens': 24}),
    (generate_delayed_copy_example, {'delay_tokens': 24}),
    (generate_repeated_key_induction_example, {'noise_pairs': 5, 'target_repetitions': 3}),
    (generate_conflicting_evidence_example, {'noise_tokens': 24, 'correct_source': 'far'}),
]:
    first = generator(seed=7, **kwargs)
    second = generator(seed=7, **kwargs)
    start, end = first.answer_span
    assert first.prompt[start:end] == first.answer
    assert first.prompt.count(first.answer) == 1
    assert first == second

window_predictions = [fixed_window_predict(example, window_tokens=32) for example in suite]
helper_rows = distance_conditioned_accuracy(suite, window_predictions, bucket_boundaries=BOUNDARIES)
torch_rows = torch_bucket_accuracy(suite, window_predictions, BOUNDARIES)

for helper_row, torch_row in zip(helper_rows, torch_rows):
    assert helper_row.label == torch_row['label']
    assert helper_row.count == torch_row['count']
    assert abs(helper_row.accuracy - torch_row['accuracy']) < 1e-7

print('All generator and metric checks passed.')


In [ ]:
oracle_predictions = [oracle_predict(example) for example in suite]
window16_predictions = [fixed_window_predict(example, window_tokens=16) for example in suite]
window32_predictions = [fixed_window_predict(example, window_tokens=32) for example in suite]
window64_predictions = [fixed_window_predict(example, window_tokens=64) for example in suite]

print('Oracle distance-conditioned accuracy')
print(format_rows(torch_bucket_accuracy(suite, oracle_predictions, BOUNDARIES)))
print()
print('Window-16 distance-conditioned accuracy')
print(format_rows(torch_bucket_accuracy(suite, window16_predictions, BOUNDARIES)))
print()
print('Window-32 distance-conditioned accuracy')
print(format_rows(torch_bucket_accuracy(suite, window32_predictions, BOUNDARIES)))
print()
print('Window-64 distance-conditioned accuracy')
print(format_rows(torch_bucket_accuracy(suite, window64_predictions, BOUNDARIES)))


## Ablations

The first ablation sweeps the visible window size `W`. The second separates tasks, because the near/far conflict task should fail differently from the single-evidence tasks: once the requested far source leaves the window, the baseline returns the nearby distractor rather than an abstention.


In [ ]:
task_names = sorted({example.task_name for example in suite})
for task_name in task_names:
    task_examples = [example for example in suite if example.task_name == task_name]
    predictions = [fixed_window_predict(example, window_tokens=32) for example in task_examples]
    rows = torch_bucket_accuracy(task_examples, predictions, BOUNDARIES)
    print(task_name)
    print(format_rows(rows))
    print()


## Assumptions

- Token distance is measured in synthetic whitespace-separated symbols rather than in a production tokenizer.
- The fixed-window baseline is a diagnostic probe, not a realistic neural language model.
- Exact-match is appropriate because the answers are symbolic single-token targets.
- The synthetic tasks aim to isolate retrieval and source selection, not open-ended reasoning or generation quality.


## Risks

- A synthetic task can still be too easy if formatting cues reveal the answer without real retrieval.
- A deterministic generator can overfit our intuition about failure modes and miss others.
- Exact-match does not capture calibrated uncertainty; a model that abstains intelligently would look wrong here.
- Distance alone does not explain every error, especially when conflicting evidence introduces a source-selection problem.


## Failure criteria

This notebook is not doing its job if any of the following happen:

- the correct answer appears multiple times in one prompt,
- changing the seed changes only superficial noise while leaking the same answer patterns,
- bucketed metrics disagree with a direct aggregation check,
- the fixed-window baseline stays flat across long distances because the task accidentally leaks a nearby shortcut,
- the analysis is used to make claims about state-of-the-art long-context models without replacing the toy baseline.


## Exercises

Work the written exercises in [`/Users/tovarishchrafa/Desktop/human-learning-machine-learning/llm-from-scratch/exercises/research/19_long_context_evaluation.md`](/Users/tovarishchrafa/Desktop/human-learning-machine-learning/llm-from-scratch/exercises/research/19_long_context_evaluation.md). The solution key lives at [`/Users/tovarishchrafa/Desktop/human-learning-machine-learning/llm-from-scratch/exercises/research/solutions/19_long_context_evaluation_solutions.md`](/Users/tovarishchrafa/Desktop/human-learning-machine-learning/llm-from-scratch/exercises/research/solutions/19_long_context_evaluation_solutions.md).


## References

- Transformer-XL frames long-range sequence modeling as a segment-recurrence problem and motivates why a fixed local context is limiting [@transformer_xl_2019].
- Longformer illustrates one sparse alternative: mostly local windows with explicit global connections for selected tokens [@longformer_2020].
- Memorizing Transformers show an external-memory retrieval path for recalling rare distant content without forcing every dependency into the immediate attention window [@memorizing_transformers_2022].
